# LAB 13 & 14 — Xây dựng mô hình Machine Learning đơn giản
## Dự đoán doanh thu (Linear Regression) & Phân nhóm khách hàng (K-Means / GMM)

**Môn học:** Big Data — Bài 13 (Trí tuệ nhân tạo và Machine Learning trong phân tích dữ liệu lớn) & Bài 14 (Phân cụm và phân nhóm khách hàng bằng Machine Learning)

---

### Mục tiêu của lab

| Phần | Bài học | Kỹ thuật | Loại học | Đầu ra |
|---|---|---|---|---|
| **A** | Bài 13 | Linear Regression | Supervised — Regression | Mô hình dự đoán doanh thu + báo cáo đánh giá |
| **B** | Bài 14 | K-Means & GMM | Unsupervised — Clustering | Dataset có cột cluster + business profile từng nhóm |

> **Lưu ý về thuật ngữ (theo đúng slide Bài 14):** phần B của lab là bài toán **phân nhóm khách hàng (customer segmentation / clustering)**, không phải "phân loại" (classification) theo nghĩa kỹ thuật — vì K-Means và GMM là các thuật toán **Unsupervised Learning**, không dùng nhãn có sẵn. Cụm từ "phân loại khách hàng" trong đề bài lab được hiểu là mục tiêu nghiệp vụ (chia khách hàng thành các nhóm để hành động), và được hiện thực bằng kỹ thuật clustering đúng như nội dung Bài 14.

### Dữ liệu sử dụng

- `market_demand.csv` — dữ liệu đơn hàng & doanh thu theo ngày (2023–2025), dùng cho **Phần A**.
- `enterprise_transactions_clean.csv` — dữ liệu giao dịch khách hàng (~60,000 giao dịch, ~7,900 khách hàng), dùng cho **Phần B** (sau khi tổng hợp thành đặc trưng RFM theo từng khách hàng).

### Cách sử dụng notebook

1. Đặt 2 file CSV nói trên **cùng thư mục** với notebook này (hoặc chỉnh lại đường dẫn trong mục 0.3).
2. Chạy lần lượt từng cell theo thứ tự, từ trên xuống dưới.
3. Các ô có nhãn **`# TODO`** là phần **bài tập** — sinh viên tự hoàn thành.
4. Cuối mỗi phần có mục **"Bài tập"** để luyện tập thêm.


---
## 0. Cài đặt & chuẩn bị môi trường

### 0.1. Thư viện cần thiết

Lab dùng các thư viện phổ biến cho Data Science: `pandas`, `numpy`, `matplotlib`, `seaborn`, `scikit-learn`.
Nếu môi trường của bạn (Google Colab / Jupyter local) chưa có sẵn, chạy cell bên dưới để cài đặt.

In [ ]:
# Bỏ dấu # ở dòng dưới nếu môi trường của bạn chưa có các thư viện này (vd: máy cá nhân)
# %pip install -q pandas numpy matplotlib seaborn scikit-learn

### 0.2. Import thư viện

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

# Cấu hình hiển thị
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.max_columns", 50)

RANDOM_STATE = 42  # cố định random_state để kết quả tái lập được (reproducible)
np.random.seed(RANDOM_STATE)

print("Đã import xong thư viện.")

### 0.3. Đường dẫn dữ liệu

Hàm `find_file()` bên dưới sẽ tự tìm file trong vài vị trí phổ biến (thư mục hiện tại, `./data/`, thư mục upload của Colab...). Nếu không tìm thấy, hãy sửa lại đường dẫn thủ công.

In [ ]:
import os

def find_file(filename, search_dirs=(".", "./data", "/mnt/user-data/uploads", "/content")):
    for d in search_dirs:
        candidate = os.path.join(d, filename)
        if os.path.exists(candidate):
            return candidate
    raise FileNotFoundError(
        f"Không tìm thấy '{filename}'. Hãy đặt file cùng thư mục với notebook "
        f"hoặc sửa lại đường dẫn trực tiếp trong biến tương ứng bên dưới."
    )

PATH_REVENUE = find_file("market_demand.csv")
PATH_TRANSACTIONS = find_file("enterprise_transactions_clean.csv")

print("Revenue data     :", PATH_REVENUE)
print("Transactions data:", PATH_TRANSACTIONS)

---
---
# PHẦN A (Bài 13) — Dự đoán doanh thu bằng Linear Regression

Áp dụng đúng **quy trình 7 bước xây dựng mô hình ML** đã học ở mục 13.3:

`Xác định bài toán → Chuẩn bị dữ liệu → Chọn feature → Chia Train/Test → Huấn luyện (Training) → Đánh giá (Evaluation) → Dự đoán (Prediction)`

**Bài toán:** dự đoán `revenue_trieu` (doanh thu, đơn vị triệu đồng) từ các biến đầu vào có sẵn theo ngày. Đây là bài toán **Regression** (dự đoán một giá trị số liên tục) — khác với **Classification** (dự đoán một nhãn/lớp) sẽ gặp lại ở Phần B.

## A.1. Xác định bài toán

- **Target (biến cần dự đoán):** `revenue_trieu`
- **Feature (biến đầu vào):** `orders` (số đơn hàng trong ngày) và các đặc trưng thời gian được tạo thêm ở bước feature engineering.

> Slide gốc minh họa với bộ feature `marketing_expense, customers, orders, discount → revenue`. Dữ liệu thực hành đi kèm (`market_demand.csv`) chỉ có `orders`, nên ta sẽ **tạo thêm feature từ cột `date`** để mô hình có nhiều thông tin hơn — đây cũng là một bước rất thực tế trong công việc thật.

In [ ]:
df_rev = pd.read_csv(PATH_REVENUE, parse_dates=["date"])
print("Kích thước dữ liệu:", df_rev.shape)
df_rev.head()

## A.2. Khám phá dữ liệu (EDA)

In [ ]:
df_rev.info()

In [ ]:
df_rev.describe()

In [ ]:
# Kiểm tra dữ liệu thiếu (missing values)
df_rev.isna().sum()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(df_rev["date"], df_rev["revenue_trieu"], color="#1f3d99", linewidth=0.8)
axes[0].set_title("Doanh thu theo ngày (triệu đồng)")
axes[1].plot(df_rev["date"], df_rev["orders"], color="#7fae1e", linewidth=0.8)
axes[1].set_title("Số đơn hàng theo ngày")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
sns.scatterplot(data=df_rev, x="orders", y="revenue_trieu", alpha=0.4)
plt.title("Mối quan hệ giữa số đơn hàng (orders) và doanh thu (revenue)")
plt.xlabel("orders (feature)")
plt.ylabel("revenue_trieu (target)")
plt.show()

print("Hệ số tương quan Pearson:", df_rev["orders"].corr(df_rev["revenue_trieu"]).round(3))

**Nhận xét:** `orders` và `revenue_trieu` có tương quan tuyến tính khá mạnh (điểm dữ liệu bám sát một đường thẳng) — đúng như khái niệm Linear Regression đã học ở mục 13.6: *"tìm một đường thẳng mô tả tốt nhất mối quan hệ giữa feature và target"*.

## A.3. Chuẩn bị dữ liệu / Feature Engineering

Từ cột `date`, ta trích thêm các đặc trưng thời gian: thứ trong tuần, tháng, và cờ cuối tuần — vì hành vi mua sắm thường khác nhau giữa ngày thường và cuối tuần, giữa các tháng trong năm.

In [ ]:
df_rev["day_of_week"] = df_rev["date"].dt.dayofweek   # 0 = Thứ Hai, 6 = Chủ Nhật
df_rev["month"] = df_rev["date"].dt.month
df_rev["is_weekend"] = (df_rev["day_of_week"] >= 5).astype(int)

df_rev.head()

## A.4. Chọn Feature & chia Train / Test

Chọn feature đầu vào và chia dữ liệu theo tỉ lệ 80/20 — 80% để huấn luyện (train), 20% để kiểm tra mô hình trên dữ liệu mô hình **chưa từng thấy** (test).

In [ ]:
feature_cols = ["orders", "day_of_week", "month", "is_weekend"]
target_col = "revenue_trieu"

X = df_rev[feature_cols]
y = df_rev[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape[0]} dòng | Test: {X_test.shape[0]} dòng")

> **Ghi chú:** vì đây là dữ liệu chuỗi thời gian (time series), trong thực tế dự báo tương lai ta thường chia theo mốc thời gian (train = quá khứ, test = giai đoạn gần nhất) thay vì chia ngẫu nhiên. Ở đây ta dùng chia ngẫu nhiên để thực hành đúng quy trình cơ bản đã học; bạn sẽ thử lại kiểu chia theo thời gian ở phần **Bài tập**.

## A.5. Huấn luyện mô hình (Training)

In [ ]:
model_lr = LinearRegression()
model_lr.fit(X_train, y_train)

print("Hệ số hồi quy (coefficients):")
for f, c in zip(feature_cols, model_lr.coef_):
    print(f"  {f:15s}: {c:8.4f}")
print(f"Intercept (hằng số): {model_lr.intercept_:.4f}")

## A.6. Đánh giá mô hình (Evaluation)

Sử dụng các chỉ số hồi quy phổ biến: **MAE**, **MSE**, **RMSE**, **R²**.

In [ ]:
y_pred = model_lr.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("===== BÁO CÁO ĐÁNH GIÁ MÔ HÌNH (Linear Regression) =====")
print(f"MAE  (sai số tuyệt đối trung bình) : {mae:8.2f} triệu đồng")
print(f"MSE  (sai số bình phương trung bình): {mse:8.2f}")
print(f"RMSE (căn bậc 2 của MSE)            : {rmse:8.2f} triệu đồng")
print(f"R^2  (tỉ lệ phương sai giải thích)  : {r2:8.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(y_test, y_pred, alpha=0.5, color="#1f3d99")
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, "r--", label="Dự đoán hoàn hảo")
axes[0].set_xlabel("Giá trị thực tế (Actual)")
axes[0].set_ylabel("Giá trị dự đoán (Predicted)")
axes[0].set_title("Actual vs Predicted")
axes[0].legend()

residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.5, color="#7fae1e")
axes[1].axhline(0, color="r", linestyle="--")
axes[1].set_xlabel("Giá trị dự đoán (Predicted)")
axes[1].set_ylabel("Phần dư (Residual = Actual - Predicted)")
axes[1].set_title("Biểu đồ phần dư (Residual Plot)")

plt.tight_layout()
plt.show()

**Cách đọc kết quả:**
- Điểm nằm càng gần đường chéo đỏ (Actual vs Predicted) → mô hình dự đoán càng chính xác.
- Residual Plot lý tưởng là các điểm phân tán ngẫu nhiên quanh đường 0, không theo một hình dạng (pattern) rõ rệt — nếu có pattern, nghĩa là mô hình tuyến tính đang bỏ sót một mối quan hệ phi tuyến trong dữ liệu.

## A.7. Dự đoán (Prediction) trên dữ liệu mới

Thử dự đoán doanh thu cho một ngày giả định (thứ Bảy, tháng 12, 1200 đơn hàng).

In [ ]:
ngay_moi = pd.DataFrame([{
    "orders": 1200,
    "day_of_week": 5,   # Thứ Bảy
    "month": 12,
    "is_weekend": 1
}])

du_doan = model_lr.predict(ngay_moi)[0]
print(f"Doanh thu dự đoán: {du_doan:.1f} triệu đồng")

## A.8. Bài tập — Phần A

Hoàn thành các bài tập sau (điền vào ô `# TODO`). Đây là phần **tự thực hành**, không có lời giải sẵn.

**Bài tập A.1 — Thêm feature "lag" (độ trễ):**
Doanh thu hôm nay thường liên quan đến doanh thu hôm qua. Hãy tạo thêm cột `revenue_lag1` (doanh thu của ngày hôm trước) làm feature, huấn luyện lại mô hình, và so sánh R² với mô hình ban đầu.
> Gợi ý: dùng `df_rev["revenue_trieu"].shift(1)`, sau đó nhớ loại bỏ dòng đầu tiên bị `NaN`.

In [ ]:
# TODO: Bài tập A.1
# Bước 1: Tạo cột revenue_lag1
# df_rev["revenue_lag1"] = ...

# Bước 2: Loại bỏ dòng NaN đầu tiên
# df_rev_lag = df_rev.dropna()

# Bước 3: Chọn feature mới (bao gồm revenue_lag1) và huấn luyện lại mô hình
# ...

# Bước 4: So sánh R^2 mô hình mới với mô hình ban đầu (r2 ở mục A.6)
# ...


**Bài tập A.2 — So sánh với Decision Tree Regressor:**
Bài 13.8 đã giới thiệu Decision Tree. Hãy huấn luyện một `DecisionTreeRegressor` (đã import sẵn ở mục 0.2) trên cùng tập `X_train, y_train`, đánh giá bằng MAE/RMSE/R² trên `X_test, y_test`, và so sánh với Linear Regression.

In [ ]:
# TODO: Bài tập A.2
# model_dt = DecisionTreeRegressor(max_depth=..., random_state=RANDOM_STATE)
# model_dt.fit(X_train, y_train)
# y_pred_dt = model_dt.predict(X_test)
# ... tính mae, rmse, r2 cho model_dt và so sánh với model_lr


**Bài tập A.3 — Nhận xét feature quan trọng:**
Dựa vào các hệ số hồi quy (`model_lr.coef_`) đã in ở mục A.5, hãy vẽ một biểu đồ cột (bar chart) thể hiện độ lớn của từng hệ số, và viết 2–3 câu nhận xét: feature nào ảnh hưởng nhiều nhất đến doanh thu? Dấu (+/-) của hệ số nói lên điều gì?

In [ ]:
# TODO: Bài tập A.3
# plt.bar(feature_cols, model_lr.coef_)
# ...

# Nhận xét của bạn (viết bằng markdown ở cell dưới hoặc comment tại đây):


---
---
# PHẦN B (Bài 14) — Phân nhóm khách hàng bằng K-Means & GMM

Áp dụng **Pipeline LAB đề xuất** ở mục 14 thực hành:

`Prepare (chọn feature, xử lý missing/outlier, scale) → Select K (Elbow, Silhouette) → Model (K-Means, GMM) → Interpret (centroid, profile, đặt tên, đề xuất hành động)`

**Bài toán:** nhóm ~7,900 khách hàng trong `enterprise_transactions_clean.csv` thành các phân khúc (segment) có hành vi tương đồng, dựa trên lịch sử giao dịch — **không dùng nhãn có sẵn** (Unsupervised Learning), khác với Phần A ở trên (Supervised Learning).

## B.1. Nạp & khám phá dữ liệu giao dịch

In [ ]:
df_tx = pd.read_csv(PATH_TRANSACTIONS, parse_dates=["ts"])
print("Kích thước dữ liệu:", df_tx.shape)
df_tx.head()

In [ ]:
df_tx.info()

In [ ]:
print("Số khách hàng (customer_id) duy nhất:", df_tx["customer_id"].nunique())
print("\nTrạng thái giao dịch (status):")
print(df_tx["status"].value_counts())
print("\nSố giao dịch bị đánh dấu outlier (la_outlier_iqr):")
print(df_tx["la_outlier_iqr"].value_counts())

**Xử lý nghiệp vụ trước khi tổng hợp:** chỉ giữ lại các giao dịch **đã hoàn tất** (`status == "HOAN_TAT"`) khi tính hành vi mua sắm — giao dịch bị hoàn trả (`HOAN_TRA`) không phản ánh đúng giá trị khách hàng thực sự mang lại.

In [ ]:
df_tx_valid = df_tx[df_tx["status"] == "HOAN_TAT"].copy()
print(f"Giữ lại {len(df_tx_valid):,} / {len(df_tx):,} giao dịch để phân tích.")

## B.2. Data Audit — theo đúng checklist slide 14.38

> *"Không dùng `customer_id` làm feature (đây là định danh, không phải hành vi) → Kiểm tra missing → Kiểm tra outlier (spending cực lớn có thể kéo centroid) → Kiểm tra trùng lặp → Sau đó mới scale và chạy Elbow/Silhouette."*

In [ ]:
# Kiểm tra missing trên các cột hành vi liên quan
cols_to_check = ["customer_id", "amount", "quantity", "ts"]
print("Tỉ lệ thiếu (%):")
print((df_tx_valid[cols_to_check].isna().mean() * 100).round(2))

In [ ]:
# Kiểm tra trùng lặp transaction_id
print("Số transaction_id trùng lặp:", df_tx_valid["transaction_id"].duplicated().sum())

## B.3. Chọn Feature & xây dựng bảng RFM theo khách hàng

Dữ liệu gốc là **giao dịch** (mỗi dòng = 1 giao dịch), nhưng clustering cần **1 dòng = 1 khách hàng**. Ta tổng hợp (`groupby`) theo `customer_id` để tạo bộ đặc trưng **RFM** — kỹ thuật kinh điển trong Customer Segmentation:

| Feature | Ý nghĩa |
|---|---|
| `recency` | Số ngày kể từ lần mua gần nhất (càng nhỏ càng "mới mua gần đây") |
| `frequency` | Tổng số giao dịch (tần suất mua) |
| `monetary` | Tổng chi tiêu (giá trị mang lại) |
| `avg_transaction` | Giá trị trung bình mỗi giao dịch |

(Tương ứng nhóm **Behavior** và **Value** ở slide 14.18 — `total_spending`, `avg_transaction`, `transaction_count`, `frequency`, `recency`.)

In [ ]:
snapshot_date = df_tx_valid["ts"].max() + pd.Timedelta(days=1)

customer_df = df_tx_valid.groupby("customer_id").agg(
    recency=("ts", lambda x: (snapshot_date - x.max()).days),
    frequency=("transaction_id", "count"),
    monetary=("amount", "sum"),
    avg_transaction=("amount", "mean"),
).reset_index()

print("Số khách hàng sau khi tổng hợp:", customer_df.shape[0])
customer_df.head()

In [ ]:
customer_df.describe()

## B.4. Xử lý Outlier & Chuẩn hóa (Preprocessing)

Theo mục 14.4 và 14.19: các thuật toán dựa trên khoảng cách (K-Means, GMM) **rất nhạy với scale** và với **outlier** — một khách hàng chi tiêu cực lớn có thể "kéo" centroid lệch hẳn khỏi số đông. Ta xử lý theo 2 bước:

1. **Outlier**: giới hạn (cap) giá trị theo phương pháp IQR thay vì xóa hẳn khách hàng (tránh mất thông tin).
2. **Scale**: dùng `StandardScaler` để đưa các feature về cùng đơn vị đo (mean=0, std=1) — bắt buộc trước khi tính khoảng cách Euclidean.

In [ ]:
def cap_outliers_iqr(series):
    """Giới hạn giá trị ngoài [Q1 - 1.5*IQR, Q3 + 1.5*IQR] về đúng biên đó."""
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return series.clip(lower, upper)

feature_cols_b = ["recency", "frequency", "monetary", "avg_transaction"]

customer_capped = customer_df.copy()
for col in feature_cols_b:
    n_outliers = ((customer_df[col] < customer_df[col].quantile(0.25) - 1.5*(customer_df[col].quantile(0.75)-customer_df[col].quantile(0.25))) |
                  (customer_df[col] > customer_df[col].quantile(0.75) + 1.5*(customer_df[col].quantile(0.75)-customer_df[col].quantile(0.25)))).sum()
    print(f"{col:18s}: {n_outliers} outlier(s) sẽ được giới hạn (cap)")
    customer_capped[col] = cap_outliers_iqr(customer_capped[col])

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(customer_capped[feature_cols_b])

print("Dữ liệu sau chuẩn hóa — mean ≈ 0, std ≈ 1:")
print(pd.DataFrame(X_scaled, columns=feature_cols_b).describe().loc[["mean", "std"]].round(3))

## B.5. Chọn số cluster K — Elbow Method & Silhouette Score

- **Elbow Method** (mục 14.5): vẽ Inertia theo từng K, tìm điểm "khuỷu tay" nơi Inertia giảm chậm lại. Lưu ý slide 14.14: *"Inertia luôn giảm khi K tăng, vì vậy không chọn K chỉ bằng Inertia nhỏ."*
- **Silhouette Score** (mục 14.5): đo mức độ một điểm "khớp" với cluster của nó, giá trị trong [-1, 1] — càng gần **+1** càng tốt.

In [ ]:
K_range = range(2, 9)
inertias = []
silhouettes = []

for k in K_range:
    km_temp = KMeans(n_clusters=k, init="k-means++", random_state=RANDOM_STATE, n_init=10)
    labels_temp = km_temp.fit_predict(X_scaled)
    inertias.append(km_temp.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels_temp))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(list(K_range), inertias, marker="o", color="#1f3d99")
axes[0].set_xlabel("Số cluster (K)")
axes[0].set_ylabel("Inertia")
axes[0].set_title("Elbow Method")

axes[1].plot(list(K_range), silhouettes, marker="o", color="#7fae1e")
axes[1].set_xlabel("Số cluster (K)")
axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Silhouette Score theo K")

plt.tight_layout()
plt.show()

for k, inertia, sil in zip(K_range, inertias, silhouettes):
    print(f"K={k}: Inertia={inertia:10.1f}  |  Silhouette={sil:.4f}")

**Chọn K:** theo mục 14.23, chọn K là sự cân bằng giữa **kỹ thuật** (Elbow hợp lý, Silhouette tốt, cluster size không quá lệch) và **nghiệp vụ** (profile giải thích được, số nhóm đủ đơn giản để hành động). Dựa trên biểu đồ trên, ta chọn `K = 4` (điều chỉnh nếu biểu đồ của bạn cho kết quả khác — dữ liệu train/test không dùng random split nên kết quả sẽ ổn định giữa các lần chạy).

In [ ]:
K = 4  # <-- điều chỉnh nếu cần, dựa trên biểu đồ Elbow/Silhouette phía trên
print(f"Số cluster được chọn: K = {K}")

## B.6. Huấn luyện K-Means

Dùng khởi tạo **K-Means++** (mục 14.15) thay vì Random — ổn định hơn, ít rơi vào local optimum hơn.

In [ ]:
kmeans = KMeans(n_clusters=K, init="k-means++", random_state=RANDOM_STATE, n_init=10)
customer_df["cluster_kmeans"] = kmeans.fit_predict(X_scaled)

print("Số khách hàng mỗi cluster (K-Means):")
print(customer_df["cluster_kmeans"].value_counts().sort_index())

## B.7. Thử GMM (Gaussian Mixture Model)

Khác với K-Means (**hard clustering** — mỗi điểm chỉ thuộc đúng 1 cluster), GMM là **soft clustering** (mục 14.6): mỗi khách hàng có một **xác suất** thuộc về từng cluster.

In [ ]:
gmm = GaussianMixture(n_components=K, random_state=RANDOM_STATE)
customer_df["cluster_gmm"] = gmm.fit_predict(X_scaled)
proba = gmm.predict_proba(X_scaled)

print("Số khách hàng mỗi cluster (GMM):")
print(customer_df["cluster_gmm"].value_counts().sort_index())

print("\nVí dụ xác suất thuộc từng cluster của 5 khách hàng đầu tiên:")
proba_df = pd.DataFrame(proba, columns=[f"P(cluster={i})" for i in range(K)])
proba_df.head().round(3)

**Soft Assignment:** giống ví dụ ở slide 14.26 — một khách hàng có thể có xác suất gần bằng nhau ở 2 cluster (nằm "ở giữa" 2 nhóm), thay vì bị gán cứng vào một nhóm duy nhất như K-Means. Hãy thử tìm một khách hàng như vậy:

In [ ]:
max_proba = proba.max(axis=1)
idx_uncertain = max_proba.argsort()[:5]  # 5 khách hàng có độ chắc chắn thấp nhất
print("5 khách hàng có xác suất phân cụm 'mập mờ' nhất (gần ranh giới giữa các nhóm):")
proba_df.iloc[idx_uncertain].round(3)

## B.8. So sánh K-Means và GMM

| | K-Means | GMM |
|---|---|---|
| Kiểu gán nhóm | Hard clustering | Soft clustering (xác suất) |
| Cơ chế | Centroid-based | Distribution-based |
| Phù hợp khi | Cluster dạng khối cầu (compact), kích thước tương đồng | Cluster có hình dạng/overlap phức tạp hơn |

In [ ]:
print("Bảng chéo (crosstab) giữa nhãn K-Means và GMM:")
print(pd.crosstab(customer_df["cluster_kmeans"], customer_df["cluster_gmm"],
                   rownames=["K-Means"], colnames=["GMM"]))

sil_kmeans = silhouette_score(X_scaled, customer_df["cluster_kmeans"])
sil_gmm = silhouette_score(X_scaled, customer_df["cluster_gmm"])
print(f"\nSilhouette Score — K-Means: {sil_kmeans:.4f}  |  GMM: {sil_gmm:.4f}")

## B.9. Từ Centroid đến Customer Profile (mục 14.30)

Centroid nằm trong không gian đã chuẩn hóa (scaled) nên khó đọc trực tiếp — cần `inverse_transform` để đưa về đơn vị gốc (VND, số giao dịch, ngày) trước khi diễn giải nghiệp vụ.

In [ ]:
centroids_scaled = kmeans.cluster_centers_
centroids_original = scaler.inverse_transform(centroids_scaled)

profile = pd.DataFrame(centroids_original, columns=feature_cols_b)
profile.index.name = "cluster_kmeans"
profile["so_khach_hang"] = customer_df["cluster_kmeans"].value_counts().sort_index().values
profile = profile.round(0)

profile

**Đặt tên nhóm theo nghiệp vụ:** dựa vào bảng profile phía trên (so sánh mỗi cluster với giá trị trung bình chung), hãy quan sát:
- Cluster nào có `monetary` và `frequency` cao, `recency` thấp → khách hàng **giá trị cao, mua thường xuyên gần đây** (High-value).
- Cluster nào có `recency` rất cao (lâu chưa quay lại) → khách hàng **có nguy cơ rời bỏ** (At-risk / Low-active).
- Các cluster còn lại thường là nhóm **Regular** (khách hàng phổ thông).

In [ ]:
overall_mean = customer_df[feature_cols_b].mean()
print("Giá trị trung bình toàn bộ khách hàng (để so sánh với từng cluster):")
print(overall_mean.round(0))

In [ ]:
# TODO: đặt tên nghiệp vụ cho từng cluster dựa trên bảng profile ở trên
# Gợi ý: chỉnh sửa dictionary bên dưới theo đúng số cluster K và đặc điểm quan sát được
cluster_names = {
    0: "TODO: đặt tên nhóm 0",
    1: "TODO: đặt tên nhóm 1",
    2: "TODO: đặt tên nhóm 2",
    3: "TODO: đặt tên nhóm 3",
}

customer_df["segment_name"] = customer_df["cluster_kmeans"].map(cluster_names)
customer_df[["customer_id", "cluster_kmeans", "segment_name"]].head()

## B.10. Trực quan hóa Cluster (mục 14.41)

Theo đúng hướng dẫn slide: dùng **2 feature quan trọng nhất** cho scatter 2D (không coi đây là toàn bộ cấu trúc dữ liệu — dữ liệu thực có 4 chiều), và **boxplot** để kiểm tra overlap/outlier giữa các cluster.

In [ ]:
plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    customer_df["frequency"], customer_df["monetary"],
    c=customer_df["cluster_kmeans"], cmap="viridis", alpha=0.6, s=20
)
plt.xlabel("frequency (số giao dịch)")
plt.ylabel("monetary (tổng chi tiêu)")
plt.title("Phân nhóm khách hàng theo Frequency vs Monetary (K-Means)")
plt.colorbar(scatter, label="Cluster")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(feature_cols_b), figsize=(16, 4))
for ax, col in zip(axes, feature_cols_b):
    sns.boxplot(data=customer_df, x="cluster_kmeans", y=col, hue="cluster_kmeans",
                ax=ax, palette="viridis", legend=False)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## B.11. Đánh giá thêm — Size, Separation, Stability (mục 14.31)

Clustering là **exploratory analysis** — ngoài Elbow/Silhouette, nên kiểm tra thêm 3 khía cạnh:
- **Size**: cluster có bị quá nhỏ / mất cân bằng không?
- **Separation**: các cluster có phân biệt đủ rõ theo feature không (đã xem ở boxplot)?
- **Stability**: đổi `random_state` thì kết quả có ổn định không?

In [ ]:
print("Kiểm tra Size — tỉ lệ % khách hàng mỗi cluster:")
print((customer_df["cluster_kmeans"].value_counts(normalize=True) * 100).round(1).sort_index())

In [ ]:
# Kiểm tra Stability: chạy lại K-Means với random_state khác, so sánh độ trùng khớp
kmeans_v2 = KMeans(n_clusters=K, init="k-means++", random_state=123, n_init=10)
labels_v2 = kmeans_v2.fit_predict(X_scaled)

print("So sánh nhãn cluster giữa 2 lần chạy với random_state khác nhau:")
print(pd.crosstab(customer_df["cluster_kmeans"], labels_v2,
                   rownames=["random_state=42"], colnames=["random_state=123"]))
print("\n→ Nếu phần lớn khách hàng vẫn rơi vào cùng một nhóm (dù đổi thứ tự nhãn số),")
print("   nghĩa là kết quả phân cụm khá ổn định (stable).")

## B.12. Sản phẩm cuối cùng (mục 14.43)

Theo đúng 3 đầu ra yêu cầu của Bài 14: **(1)** Notebook Customer Segmentation hoàn chỉnh (K-Means + GMM), **(2)** Dataset có cột cluster (giữ `customer_id` để nhận diện), **(3)** Business Profile — bảng profile từng nhóm kèm tên segment + đề xuất.

In [ ]:
# (2) Dataset có cột cluster — giữ customer_id để nhận diện khách hàng
output_customers = customer_df[[
    "customer_id", "recency", "frequency", "monetary", "avg_transaction",
    "cluster_kmeans", "cluster_gmm", "segment_name"
]]

output_customers.to_csv("customer_segments_output.csv", index=False)
print("Đã lưu 'customer_segments_output.csv' —", output_customers.shape[0], "khách hàng.")
output_customers.head()

In [ ]:
# (3) Business Profile — bảng tổng hợp từng nhóm
business_profile = profile.copy()
business_profile["segment_name"] = [cluster_names.get(i, "") for i in business_profile.index]
business_profile["% khach_hang"] = (customer_df["cluster_kmeans"].value_counts(normalize=True).sort_index() * 100).round(1).values

business_profile

## B.13. Bài tập — Phần B

**Bài tập B.1 — Thử các giá trị K khác nhau:**
Chạy lại K-Means với `K = 3` và `K = 5`, so sánh Silhouette Score và số lượng khách hàng mỗi cluster với kết quả `K = 4` ở trên. Bạn chọn K nào là hợp lý nhất? Vì sao?

In [ ]:
# TODO: Bài tập B.1
# for k_thu in [3, 5]:
#     km_thu = KMeans(n_clusters=k_thu, init="k-means++", random_state=RANDOM_STATE, n_init=10)
#     labels_thu = km_thu.fit_predict(X_scaled)
#     sil_thu = silhouette_score(X_scaled, labels_thu)
#     print(f"K={k_thu}: Silhouette={sil_thu:.4f}")
#     print(pd.Series(labels_thu).value_counts().sort_index())


**Bài tập B.2 — Thêm feature mới:**
Từ `df_tx_valid`, hãy tính thêm feature `online_ratio` = tỉ lệ giao dịch qua kênh `Online` trên tổng số giao dịch của mỗi khách hàng (cột `channel`). Thêm feature này vào `customer_df`, chuẩn hóa lại, và chạy lại K-Means. Cluster mới có gì khác so với ban đầu?
> Gợi ý: `df_tx_valid.groupby("customer_id")["channel"].apply(lambda x: (x == "Online").mean())`

In [ ]:
# TODO: Bài tập B.2
# online_ratio = df_tx_valid.groupby("customer_id")["channel"].apply(lambda x: (x == "Online").mean())
# customer_df["online_ratio"] = customer_df["customer_id"].map(online_ratio)
# ... chuẩn hóa lại và chạy K-Means với feature mới


**Bài tập B.3 — Đề xuất hành động kinh doanh:**
Với mỗi cluster đã đặt tên ở mục B.9, hãy viết 1–2 câu đề xuất hành động marketing/CRM cụ thể (ví dụ: nhóm High-value → chương trình khách hàng thân thiết; nhóm At-risk → email khuyến mãi kích hoạt lại). Đây chính là bước **"Design → Execute"** trong vòng đời Customer Segmentation ở mục 14.34.

In [ ]:
# TODO: Bài tập B.3 — viết đề xuất hành động cho từng cluster
de_xuat_hanh_dong = {
    0: "TODO: đề xuất hành động cho nhóm 0",
    1: "TODO: đề xuất hành động cho nhóm 1",
    2: "TODO: đề xuất hành động cho nhóm 2",
    3: "TODO: đề xuất hành động cho nhóm 3",
}
for cluster_id, action in de_xuat_hanh_dong.items():
    ten = cluster_names.get(cluster_id, "?")
    print(f"Cluster {cluster_id} ({ten}): {action}")

---
---
## Tổng kết bài Lab

| | Phần A — Dự đoán doanh thu | Phần B — Phân nhóm khách hàng |
|---|---|---|
| Bài học | Bài 13 | Bài 14 |
| Loại học | Supervised Learning | Unsupervised Learning |
| Bài toán | Regression | Clustering |
| Thuật toán | Linear Regression | K-Means, GMM |
| Đánh giá | MAE, MSE, RMSE, R² | Silhouette Score, Elbow, Business Profile |
| Sản phẩm | Mô hình dự đoán + báo cáo đánh giá | Dataset có cột cluster + business profile |

### Checklist nộp bài

- [ ] Notebook chạy được từ đầu đến cuối, không lỗi (Kernel → Restart & Run All)
- [ ] Phần A: đã hoàn thành Bài tập A.1, A.2, A.3
- [ ] Phần B: đã hoàn thành Bài tập B.1, B.2, B.3 (bao gồm đặt tên cluster ở mục B.9)
- [ ] File `customer_segments_output.csv` được sinh ra sau khi chạy notebook
- [ ] Đã viết nhận xét / đề xuất hành động bằng lời văn của chính bạn (không để nguyên TODO)

**Bài tiếp theo:** Graph Analytics và mô hình hóa Big Data + AI.
